# Project 65 — Local Hallucination Audit
## Claim Extraction → Source Verification → Scoring Dashboard

**Stack:** LangChain · Ollama · Pydantic · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Ground Truth Sources

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.0)

sources = {
    "company_report": (
        "Acme Corp revenue was $2.3M in Q3 2024, up 15% from Q2. "
        "The company has 150 employees across 3 offices in NYC, London, and Tokyo. "
        "CEO is Jane Smith, appointed in 2021. CTO is Bob Chen."
    ),
    "product_specs": (
        "Widget X weighs 2.5 kg, costs $49.99 retail. Available in blue and red. "
        "Battery life is 8 hours. Launched March 2024. Uses USB-C charging. "
        "IP67 water resistance rating."
    ),
    "policy_doc": (
        "Employees get 20 PTO days per year. Health insurance covers 80% of premiums. "
        "401k match is 4% of salary. Remote work allowed 3 days per week. "
        "Parental leave is 12 weeks paid."
    ),
}
print(f"Source documents: {len(sources)}")
for k, v in sources.items():
    print(f"  {k}: {len(v)} chars")

## Step 2 — Generate Answers to Audit

In [ ]:
# Generate answers with varying temperatures (to induce hallucinations)
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based on the source. Include specific numbers and details."),
    ("human", "Source: {source}\n\nQuestion: {question}")
])

audit_cases = [
    {"q": "What was Acme Corp's Q3 revenue and growth?", "source": "company_report",
     "temp": 0.1},
    {"q": "Tell me about Acme Corp", "source": "company_report",
     "temp": 0.9},  # Higher temp = more hallucination risk
    {"q": "What are Widget X's specifications?", "source": "product_specs",
     "temp": 0.1},
    {"q": "Describe the Widget X product", "source": "product_specs",
     "temp": 0.8},
    {"q": "What are the employee benefits?", "source": "policy_doc",
     "temp": 0.1},
    {"q": "Summarize the company benefits package", "source": "policy_doc",
     "temp": 0.7},
]

answers = []
for case in audit_cases:
    gen = ChatOllama(model="qwen3:8b", temperature=case["temp"])
    chain = qa_prompt | gen | StrOutputParser()
    answer = chain.invoke({"source": sources[case["source"]], "question": case["q"]})
    answers.append({**case, "answer": answer})
    print(f"  T={case['temp']} | {case['q'][:40]}... → {len(answer)} chars")

## Step 3 — Claim Extraction & Verification

In [ ]:
class Claim(BaseModel):
    claim_text: str
    status: str = Field(description="supported, contradicted, unverifiable")
    source_evidence: str = Field(description="Quote from source or 'not found'")
    severity: str = Field(description="minor, major, critical")

class AuditResult(BaseModel):
    total_claims: int
    supported: int
    contradicted: int
    unverifiable: int
    hallucination_score: float = Field(ge=0, le=1)
    claims: list[Claim]

auditor = llm.with_structured_output(AuditResult)

all_results = []
for case in answers:
    source = sources[case["source"]]
    result = auditor.invoke(
        f"SOURCE (ground truth):\n{source}\n\n"
        f"ANSWER TO AUDIT:\n{case['answer']}\n\n"
        f"Extract EVERY factual claim. Verify each against the source."
    )
    all_results.append({"case": case, "audit": result})
    print(f"\nQ: {case['q'][:40]}... (temp={case['temp']})")
    print(f"  Claims: {result.total_claims} | Hallucination: {result.hallucination_score:.0%}")
    for c in result.claims:
        icon = {"supported":"✓","contradicted":"✗","unverifiable":"?"}[c.status]
        print(f"    {icon} [{c.status}] {c.claim_text[:60]}")

## Step 4 — Hallucination Dashboard

In [ ]:
rows = []
for r in all_results:
    rows.append({
        "question": r["case"]["q"][:40],
        "temperature": r["case"]["temp"],
        "total_claims": r["audit"].total_claims,
        "supported": r["audit"].supported,
        "contradicted": r["audit"].contradicted,
        "unverifiable": r["audit"].unverifiable,
        "hallucination_score": r["audit"].hallucination_score,
    })

dashboard = pd.DataFrame(rows)
print("HALLUCINATION AUDIT DASHBOARD")
print("=" * 70)
print(dashboard.to_string(index=False))

print(f"\nKEY METRICS:")
print(f"  Avg hallucination score: {dashboard['hallucination_score'].mean():.0%}")
print(f"  Worst case: {dashboard['hallucination_score'].max():.0%}")
print(f"  Total claims audited: {dashboard['total_claims'].sum()}")
print(f"  Total contradicted: {dashboard['contradicted'].sum()}")

# Temperature correlation
by_temp = dashboard.groupby("temperature")["hallucination_score"].mean()
print(f"\nHallucination by temperature:")
for temp, score in by_temp.items():
    print(f"  T={temp}: {score:.0%}")

## What You Learned
- **Claim-level fact checking** against source documents
- **Temperature impact** on hallucination rates
- **Hallucination scoring** with severity classification
- **Audit dashboard** for systematic quality tracking